# **Интерпритация DeepCA**

In [ ]:
import os
import torch
# Настройка для избежания фрагментации памяти
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Мониторинг памяти
def print_memory_usage():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")

# Очистка памяти перед началом
torch.cuda.empty_cache()
print_memory_usage()

## *Библиотеки*

In [ ]:
import torchvision.transforms as T
import numpy as np
import os
import torch
from common import *
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from Reconstruction_coronary_arteria.open_json import GeneratedDataset
import cv2
from scipy.ndimage import map_coordinates, zoom, label  # Добавляем импорт zoom
import PIL 
from tqdm import tqdm  # для красивого прогресса

## **Воссоздание воксельной структуры обратного преобразования**

In [ ]:
# Пути к данным
images_dir = "/home/alexus/Desktop/For_Generator/data0_1000/content/vessel_tree_generator/data/test/images/test/"
labels_dir = "/home/alexus/Desktop/For_Generator/data0_1000/content/vessel_tree_generator/data/test/labels/test/"
info_dir = "/home/alexus/Desktop/For_Generator/data0_1000/content/vessel_tree_generator/data/test/info/"

# Трансформации для проекций
def mean_channel(x):
    return x.mean(axis=0)[None, :]

def distance_transform(image_mask):
    image_mask = np.asarray(image_mask)[0]
    binary_mask = (image_mask > np.mean(image_mask)).astype(np.uint8)
    distance = cv2.distanceTransform(binary_mask, cv2.DIST_L2, 5)
    return torch.tensor(image_mask + distance[None, :])

# images_transform = T.Compose([
    # T.ToTensor(),
    # T.Lambda(lambda x: x.mean(axis=0)[None, :] if x.shape[0] > 1 else x),
    # T.Resize((128, 128)),
    # T.Lambda(lambda x: x * 5.0),  # Масштабируем обратно до 0.25–5.0
# ])
# 
# Загрузка данных
# data = GeneratedDataset(images_dir, labels_dir, info_dir, images_transform=images_transform)
# sample_index = 0 #  индекс проекции
# images, model, theta, phi = data[sample_index]


# Трансформация
images_transform = T.Compose([
    T.ToTensor(),
    T.Resize((128, 128)),
    T.Lambda(lambda x: x * 5.0),
])

data = GeneratedDataset(images_dir, labels_dir, info_dir, images_transform=images_transform)
sample_index = 4
sample = data[sample_index]



In [ ]:
print(f'max значение sample[1][0] = {sample[1][0].max().item()}\nmin значение sample[1][0] = {sample[1][0].min().item()}')

In [ ]:
"""
data - [index образца 0-19][0 - images, 1 - 3D model (300 points), 2 - theta, 3 - phi][index proj = 0-1]
"""
plt.figure(figsize=(15,15))
plt.subplot(1,4,1)
plt.imshow(sample[0][0])
plt.axis('off')
plt.subplot(1,4,2)
plt.imshow(sample[0][4])
plt.axis('off')

In [ ]:
class Backprojection:
    def __init__(self, sid, pixel_spacing, volume_size, volume_spacing, dso=None, img_dim=128):
        self.sid = sid
        self.pixel_spacing = pixel_spacing
        self.volume_size = volume_size
        self.volume_spacing = volume_spacing
        self.dso = dso if dso is not None else sid * 0.75
        self.img_dim = img_dim
        self.principal_point = np.array([img_dim/2, img_dim/2])
        self.distance_detector_to_iso = sid - self.dso
        self.coord_system_changer = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) #[0, -1, 0], [1, 0, 0], [0, 0, -1] перевод в правую ДСК

    def get_projection_matrix(self, theta, phi): # Преобразование координат (поворот) в плоскости проекции
        theta = float(theta)
        phi = float(phi)
        theta_rad = np.radians(theta)
        phi_rad = np.radians(phi)

        """
        C = Vector_proj_coord = [-y, x, -z] - Медицинские координаты (Левая ДСК) x,y,z -> -y, x, -z - НЕ РАБОТАЕТ! 
        ТЕПЕРЬ С - ЭТО ПДСК [x,y,z], Саша, разберись почему это так стало работать!!! 
        Rotation_AP1 = C @ Rotate_around_Z @ C^-1, 
        где С^1 - обратная матрица, Rotate_around_Z - Матрица поворота вокруг Z (theta)
        Тем самым получаем поворот вокруг Z, но не в текущей системе координат, а в другой системе, которая определяется матрицей С.

        Rotation_AP2 = C @ Rotate_around_X @ C^-1,
        где Rotate_around_X - Матрица поворота вокруг Х (phi)
        """
        Rotation_AP1 = self.coord_system_changer @ np.array([
            [np.cos(theta_rad), -np.sin(theta_rad), 0],
            [np.sin(theta_rad), np.cos(theta_rad), 0],
            [0, 0, 1]
        ]) @ np.linalg.inv(self.coord_system_changer)

        Rotation_AP2 = self.coord_system_changer @ np.array([
            [1, 0, 0],
            [0, np.cos(phi_rad), np.sin(phi_rad)],
            [0, -np.sin(phi_rad), np.cos(phi_rad)]
        ]) @ np.linalg.inv(self.coord_system_changer)

        Rotation_AP3 = np.eye(3) # ед. матрица 3х3
        R_total = Rotation_AP1 @ Rotation_AP2 @ Rotation_AP3

        v_sensor = R_total @ np.array([0, 0, self.distance_detector_to_iso]) # вектор указывающий на точку приёмника
        v_source = -v_sensor / self.distance_detector_to_iso * self.dso # вектор указывающий на точку источника

        z_axis = (v_sensor - v_source) / np.linalg.norm(v_sensor - v_source) # это направление луча от источника к сенсору, т.е. главная ось камеры (оптическая ось).
        # Нормализация делает z_axis единичным вектором.
        x_axis = np.array([1, 0, 0])
        if np.abs(np.dot(z_axis, x_axis)) > 0.99:
            x_axis = np.array([0, 1, 0]) # Если z_axis почти коллинеарен оси x, выбирается другая ось, чтобы избежать численных проблем при ортогонализации.
        x_axis = x_axis - np.dot(x_axis, z_axis) * z_axis # Ортогонализация: мы "вычёркиваем" проекцию x_axis на z_axis, чтобы получить вектор, перпендикулярный z_axis.
        x_axis = x_axis / np.linalg.norm(x_axis)
        y_axis = np.cross(z_axis, x_axis)
        y_axis = y_axis / np.linalg.norm(y_axis)

        R = np.vstack((x_axis, y_axis, z_axis)).T # это 3×3 матрица поворота из мировой системы координат → в систему камеры
        t = -v_source # вектор смещения (переноса камеры)
        Rt = np.hstack((R, t.reshape(3, 1)))

        focal_length = self.sid / self.pixel_spacing # Делим  расстояние от источника до детектора на размер пикселя — получаем фокусное расстояние в пикселях
        K = np.array([
            [focal_length, 0, self.principal_point[0]],
            [0, focal_length, self.principal_point[1]],
            [0, 0, 1]
        ])
        """
        K — матрица внутренних параметров камеры (intrinsics)
        Она определяет:фокусное расстояние и положение главной точки (оптического центра) в пикселях
        """
        P = K @ Rt # итоговая матрица камеры 3×4, которая объединяет внутренние и внешние параметры:
        return P, v_source, v_sensor, R


In [ ]:
def create_shadow_cone(projection, theta, phi, volume_shape=(128, 128, 128), sid=1200.0, pixel_spacing=1.4, dso=750.0, img_dim=128):
    D_x, D_y, D_z = volume_shape
    projection = projection.clone().detach()
    threshold = 1
    # Бинарная маска для визуализации
    projection_binary = (projection > 1).float()

    # Создаём воксельное пространство, расширяем по X и Y
    vol = torch.zeros(volume_shape, dtype=torch.float32, device=projection.device)
    volume_spacing = 0.35
    x = torch.linspace(-100.0, 100.0, D_x, device=projection.device)  # Расширяем X
    y = torch.linspace(-100.0, 100.0, D_y, device=projection.device)  # Расширяем Y
    z = torch.linspace(-200.0, 200.0, D_z, device=projection.device)  # Z от -200 до 200 мм

    # Получаем геометрию камеры
    backprojector = Backprojection(sid, pixel_spacing, volume_shape, volume_spacing, dso, img_dim)
    P, v_source, v_sensor, R = backprojector.get_projection_matrix(theta, phi)

    # Преобразуем в тензоры
    v_source = torch.tensor(v_source, dtype=torch.float32, device=projection.device)
    v_sensor = torch.tensor(v_sensor, dtype=torch.float32, device=projection.device)
    R = torch.tensor(R, dtype=torch.float32, device=projection.device)

    # Определяем координаты пикселей проекции на сенсоре
    u_coords, v_coords = torch.where(projection > 1)
    intensities = projection[u_coords, v_coords]

    # Преобразуем u, v в 3D-координаты на сенсоре
    u_physical = (u_coords.float() - img_dim/2) * pixel_spacing
    v_physical = (v_coords.float() - img_dim/2) * pixel_spacing
    sensor_points = torch.stack([
        u_physical,
        v_physical,
        torch.zeros_like(u_physical)
    ], dim=1)

    # Поворачиваем точки сенсора в глобальную систему координат
    sensor_points = (R @ sensor_points.T).T + v_sensor

    # Для каждой точки на сенсоре строим луч к источнику
    directions = v_source - sensor_points
    directions = directions / torch.norm(directions, dim=1, keepdim=True)

    # Проходим по лучу через воксельное пространство
    t_max = 3000.0  # Увеличиваем расстояние
    t_steps = torch.linspace(0, t_max, 3000, device=projection.device)  # Больше шагов

    for i in range(len(sensor_points)):
        points_along_ray = sensor_points[i] + directions[i] * t_steps[:, None]
        
        # Преобразуем точки в воксельные индексы
        x_idx = ((points_along_ray[:, 0] - (-100.0)) / (200.0 / (D_x - 1))).long()
        y_idx = ((points_along_ray[:, 1] - (-100.0)) / (200.0 / (D_y - 1))).long()
        z_idx = ((points_along_ray[:, 2] - (-200.0)) / (400.0 / (D_z - 1))).long()

        # Проверяем, что индексы в пределах объёма
        valid = (x_idx >= 0) & (x_idx < D_x) & (y_idx >= 0) & (y_idx < D_y) & (z_idx >= 0) & (z_idx < D_z)
        


        x_idx, y_idx, z_idx = x_idx[valid], y_idx[valid], z_idx[valid]

        # Заполняем воксели
        vol[x_idx, y_idx, z_idx] = intensities[i]

    return vol, v_source, v_sensor, sensor_points, directions, projection_binary

def visualize_angiograph(projection1, projection2, theta, phi, volume_shape=(128, 128, 128)):
    # Создаём тени для обеих проекций
    vol1, v_source1, v_sensor1, sensor_points1, directions1, proj_binary1 = create_shadow_cone(projection1, theta[0], phi[0], volume_shape)
    vol2, v_source2, v_sensor2, sensor_points2, directions2, proj_binary2 = create_shadow_cone(projection2, theta[1], phi[1], volume_shape)
    D_x, D_y, D_z = volume_shape
    # Пересечение теней
    backprojection = (vol1 > 0) & (vol2 > 0)
    backprojection = backprojection.float()


    # Plotly 1: Сензоры, источники и лучи
    fig1 = go.Figure()

    # 1. Источники (v_source)
    fig1.add_trace(go.Scatter3d(
        x=[v_source1[0].item(), v_source2[0].item()],
        y=[v_source1[1].item(), v_source2[1].item()],
        z=[v_source1[2].item(), v_source2[2].item()],
        mode='markers',
        marker=dict(size=10, color='red'),
        name='Sources'
    ))

    # 2. Сензоры (v_sensor) как плоскости
    sensor_size = 128 * 1.4 / 2
    for i, (v_sensor, projection) in enumerate([(v_sensor1, proj_binary1), (v_sensor2, proj_binary2)]):
        x = [v_sensor[0].item() - sensor_size, v_sensor[0].item() + sensor_size]
        y = [v_sensor[1].item() - sensor_size, v_sensor[1].item() + sensor_size]
        z = [v_sensor[2].item(), v_sensor[2].item()]
        fig1.add_trace(go.Surface(
            x=x,
            y=y,
            z=np.array(z)[None, :],
            surfacecolor=projection.cpu().numpy(),
            colorscale='Gray',
            showscale=False,
            opacity=0.7,
            name=f'Sensor {i+1}'
        ))

    # 3. Лучи (ограничим до 50)
    num_rays = 200
    for i in range(min(num_rays, len(sensor_points1))):
        start = sensor_points1[i].cpu().numpy()
        end = v_source1.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='blue', width=2),
            name='Ray 1' if i == 0 else None,
            showlegend=(i == 0)
        ))
    for i in range(min(num_rays, len(sensor_points2))):
        start = sensor_points2[i].cpu().numpy()
        end = v_source2.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='green', width=2),
            name='Ray 2' if i == 0 else None,
            showlegend=(i == 0)
        ))

    fig1.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="Angiograph: Sensors, Sources, and Rays"
    )
    fig1.show()

    # Plotly 2: Только 3D-реконструкция
    fig2 = go.Figure()

    non_zero = torch.where(backprojection > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1][indices].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2][indices].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0
    else:
        x = non_zero[0].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0

    fig2.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=2, color='purple', opacity=0.5),
        name='Reconstruction'
    ))

    fig2.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="3D Reconstruction"
    )
    fig2.show()


projection1 = sample[0][0].cuda()
projection2 = sample[0][4].cuda()
theta = sample[2]
phi = [90 + sample[3][0], 90 + sample[3][1]]

print(f"Using angles: theta={theta}, phi={phi}")

visualize_angiograph(projection1, projection2, theta, phi, volume_shape=[128, 128, 128])

## **3D Model**

In [ ]:
import plotly.graph_objects as go
from typing import List
from scipy.ndimage import zoom
import torch.nn.functional as F
def create_output_model(output_3d_model: torch.Tensor, volume_shape: List[int], target_physical_range: dict):
    """
    Создаёт воксельный объём Ground Truth с учётом поворотов и масштабирования для соответствия Backprojection.

    Args:
        output_3d_model: Входной тензор с координатами и радиусами (..., 4).
        volume_shape: Размер воксельного объёма (D_x, D_y, D_z).
        target_physical_range: Словарь с физическими диапазонами Backprojection, например,
                              {'x': (min_x, max_x), 'y': (min_y, max_y), 'z': (min_z, max_z)}.
    """
    with torch.no_grad():
        output_3d_model = output_3d_model.clone()
        D_x, D_y, D_z = volume_shape

        # Извлекаем физические диапазоны
        x_min, x_max = target_physical_range['x']
        y_min, y_max = target_physical_range['y']
        z_min, z_max = target_physical_range['z']
        x_range_physical = x_max - x_min
        y_range_physical = y_max - y_min
        z_range_physical = z_max - z_min

        # Центрируем координаты
        mean_coords = output_3d_model[..., :3].mean(dim=(0, 1))
        output_3d_model[..., :3] -= mean_coords

        # Поворот на 90° вокруг X: Z → Y, Y → -Z
        x_coords = output_3d_model[..., 0]
        y_coords = output_3d_model[..., 2]
        z_coords = -output_3d_model[..., 1]

        # Поворот на 180° вокруг X: Y → -Y, Z → -Z
        x_final_rotated = -z_coords
        y_final_rotated = x_coords
        z_final_rotated = y_coords

        # Масштабируем координаты в индексы вокселей
        x_range = x_final_rotated.max() - x_final_rotated.min()
        y_range = y_final_rotated.max() - y_final_rotated.min()
        z_range = z_final_rotated.max() - z_final_rotated.min()

        scale_x = x_range_physical / x_range if x_range != 0 else 1.0
        scale_y = y_range_physical / y_range if y_range != 0 else 1.0
        scale_z = z_range_physical / z_range if z_range != 0 else 1.0

        x_scaled = x_final_rotated * scale_x
        y_scaled = y_final_rotated * scale_y
        z_scaled = z_final_rotated * scale_z

        # Преобразуем в индексы вокселей
        output_3d_model[..., 0] = ((x_scaled - x_min) / (x_range_physical / (D_x - 1))).int().clip(0, D_x - 1)
        output_3d_model[..., 1] = ((y_scaled - y_min) / (y_range_physical / (D_y - 1))).int().clip(0, D_y - 1)
        output_3d_model[..., 2] = ((z_scaled - z_min) / (z_range_physical / (D_z - 1))).int().clip(0, D_z - 1)

        T = torch.zeros(volume_shape, dtype=torch.float32, device=output_3d_model.device)

        for group in range(output_3d_model.shape[0]):
            x = output_3d_model[group, :, 0].long()
            y = output_3d_model[group, :, 1].long()
            z = output_3d_model[group, :, 2].long()
            R = output_3d_model[group, :, 3] * 10
            for i in range(len(x)):
                radius = int((R[i] * 12).clamp(2, 15))
                x_grid, y_grid, z_grid = torch.meshgrid(
                    torch.arange(-radius, radius + 1, dtype=torch.long, device=output_3d_model.device),
                    torch.arange(-radius, radius + 1, dtype=torch.long, device=output_3d_model.device),
                    torch.arange(-radius, radius + 1, dtype=torch.long, device=output_3d_model.device)
                )
                mask = (x_grid ** 2 + y_grid ** 2 + z_grid ** 2 <= radius ** 2)
                x_shifted = (x[i] + x_grid[mask]).long()
                y_shifted = (y[i] + y_grid[mask]).long()
                z_shifted = (z[i] + z_grid[mask]).long()

                valid = (x_shifted >= 0) & (x_shifted < D_x) & (y_shifted >= 0) & (y_shifted < D_y) & (z_shifted >= 0) & (z_shifted < D_z)
                x_valid = x_shifted[valid]
                y_valid = y_shifted[valid]
                z_valid = z_shifted[valid]

                insertion_index = T[x_valid, y_valid, z_valid] == 0
                average_index = T[x_valid, y_valid, z_valid] != 0

                T[x_valid, y_valid, z_valid] += R[i] * insertion_index
                avg = (R[i] + T[x_valid, y_valid, z_valid]) / 2
                T[x_valid, y_valid, z_valid] += (avg - T[x_valid, y_valid, z_valid]) * average_index

        return T

def get_3d_tensors(projection1,projection2,correct_3d_geometry  ,tensor_output_shape=[128,128,128]):
    """
    Из двух двумерных проекций `projection1` (1,128,128), `projection2` (1,128,128) строит трехмерный тензор размером `tensor_output_shape` который содержит в себе тени от
    двумерных проекций.

    Из информации о ветках `correct_3d_geometry` (4,300,4) строит 3д тензор

    Возвращает два 3д тенозора размером `volume_shape`, первый из которых - тензор теней, второй - тензор с 3д геометрией
    """

    #phi1, phi2, theta1, theta2 = -40.0, 75.0, -10.0, -10.0 # какие здесь тетта и фи, вдоль каких осей ориентированы? 
    
    D_x, D_y, D_z = tensor_output_shape
    # Убедимся, что проекция — бинарная (0 или 1)
    #projection = (projection > 0).float()  # На случай, если остались небольшие значения
    # Масштабируем проекцию до размеров D_x, D_y

    projection1 = zoom(projection1.cpu(), (D_x / projection1.shape[0], D_y / projection1.shape[1]), order=1)
    projection2 = zoom(projection2.cpu(), (D_x / projection2.shape[0], D_y / projection2.shape[1]), order=1)

    projection1=torch.tensor(projection1).to(projection1.device)
    projection2=torch.tensor(projection2).to(projection2.device)

    T = create_output_model(correct_3d_geometry.to(projection1.device), volume_shape = tensor_output_shape)
    return T
T = get_3d_tensors(sample[0][0].cuda(),sample[0][4].cuda(),sample[1].cuda())



In [ ]:
def visualize_3d_volume(volume, title="3D Reconstruction"):
    nonzero_indices = (volume > 0.01).nonzero(as_tuple=False)
    if len(nonzero_indices) > 0:
        if len(nonzero_indices) > 10000:
            indices = torch.randperm(len(nonzero_indices))[:10000]
            nonzero_indices = nonzero_indices[indices]
        values = volume[nonzero_indices[:, 0], nonzero_indices[:, 1], nonzero_indices[:, 2]]
        x_coords = nonzero_indices[:, 0].tolist()
        y_coords = nonzero_indices[:, 1].tolist()
        z_coords = nonzero_indices[:, 2].tolist()
        radii = values.tolist()
        fig = go.Figure(data=[go.Scatter3d(
            x=x_coords, y=y_coords, z=z_coords, mode='markers',
            marker=dict(size=3, color=radii, colorscale='Viridis', opacity=0.8, colorbar=dict(title="Value"))
        )])
        fig.update_layout(
            title=title,
            scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode='cube'),
            width=700, height=700,
            template="plotly_dark"
        )
        fig.show()
    else:
        print(f"No significant nonzero values (> 0.001) in {title} to visualize!")

# Визуализация
print("Visualizing target_3d...")
visualize_3d_volume(T.squeeze().squeeze(), title="Original 3D Target")


## **3D + Backprojections**

In [ ]:
def visualize_angiograph(projection1, projection2, theta, phi, correct_3d_geometry, volume_shape=(256, 256, 256)):
    vol1, v_source1, v_sensor1, sensor_points1, directions1, proj_binary1 = create_shadow_cone(projection1, theta[0], phi[0], volume_shape)
    vol2, v_source2, v_sensor2, sensor_points2, directions2, proj_binary2 = create_shadow_cone(projection2, theta[1], phi[1], volume_shape)

    backprojection = (vol1 > 0) & (vol2 > 0)
    backprojection = backprojection.float()
    print(f"Non-zero voxels in backprojection: {(backprojection > 0).sum().item()}")

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter3d(
        x=[v_source1[0].item(), v_source2[0].item()],
        y=[v_source1[1].item(), v_source2[1].item()],
        z=[v_source1[2].item(), v_source2[2].item()],
        mode='markers',
        marker=dict(size=10, color='red'),
        name='Sources'
    ))

    sensor_size = 128 * 1.4 / 2 
    for i, (v_sensor, projection) in enumerate([(v_sensor1, proj_binary1), (v_sensor2, proj_binary2)]):
        x = [v_sensor[0].item() - sensor_size, v_sensor[0].item() + sensor_size]
        y = [v_sensor[1].item() - sensor_size, v_sensor[1].item() + sensor_size]
        z = [v_sensor[2].item(), v_sensor[2].item()]
        fig1.add_trace(go.Surface(
            x=x,
            y=y,
            z=np.array(z)[None, :],
            surfacecolor=projection.cpu().numpy(),
            colorscale='Gray',
            showscale=False,
            opacity=0.7,
            name=f'Sensor {i+1}'
        ))

    num_rays = 50
    for i in range(min(num_rays, len(sensor_points1))):
        start = sensor_points1[i].cpu().numpy()
        end = v_source1.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='blue', width=2),
            name='Ray 1' if i == 0 else None,
            showlegend=(i == 0)
        ))
    for i in range(min(num_rays, len(sensor_points2))):
        start = sensor_points2[i].cpu().numpy()
        end = v_source2.cpu().numpy()
        fig1.add_trace(go.Scatter3d(
            x=[start[0], end[0]],
            y=[start[1], end[1]],
            z=[start[2], end[2]],
            mode='lines',
            line=dict(color='green', width=2),
            name='Ray 2' if i == 0 else None,
            showlegend=(i == 0)
        ))

    fig1.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="Angiograph: Sensors, Sources, and Rays"
    )
    fig1.show()

    fig2 = go.Figure()

    D_x, D_y, D_z = volume_shape
    non_zero = torch.where(backprojection > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy() * (300.0 / (D_x - 1)) - 150.0
        y = non_zero[1][indices].cpu().numpy() * (300.0 / (D_y - 1)) - 150.0
        z = non_zero[2][indices].cpu().numpy() * (400.0 / (D_z - 1)) - 300.0
    else:
        x = non_zero[0].cpu().numpy() * (300.0 / (D_x - 1)) - 150.0
        y = non_zero[1].cpu().numpy() * (300.0 / (D_y - 1)) - 150.0
        z = non_zero[2].cpu().numpy() * (400.0 / (D_z - 1)) - 300.0

    fig2.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=2, color='purple', opacity=0.5),
        name='Reconstruction'
    ))

    fig2.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="3D Reconstruction"
    )
    fig2.show()

    fig3 = go.Figure()

    backprojection_resized = F.interpolate(backprojection.unsqueeze(0).unsqueeze(0), size=(128, 128, 128), mode='nearest').squeeze()

    non_zero = torch.where(backprojection_resized > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy()
        y = non_zero[1][indices].cpu().numpy()
        z = non_zero[2][indices].cpu().numpy()
    else:
        x = non_zero[0].cpu().numpy()
        y = non_zero[1].cpu().numpy()
        z = non_zero[2].cpu().numpy()

    x_mm = x * (300.0 / 127) - 150.0
    y_mm = y * (300.0 / 127) - 150.0
    z_mm = z * (400.0 / 127) - 300.0

    fig3.add_trace(go.Scatter3d(
        x=x_mm, y=y_mm, z=z_mm,
        mode='markers',
        marker=dict(size=2, color='purple', opacity=0.5),
        name='Reconstruction'
    ))

    # Оригинальная 3D-геометрия (T)
    T = get_3d_tensors(projection1, projection2, correct_3d_geometry, tensor_output_shape=[128, 128, 128])
    nonzero_indices = (T > 0.01).nonzero(as_tuple=False)
    if len(nonzero_indices) > max_points:
        indices = torch.randperm(len(nonzero_indices))[:max_points]
        nonzero_indices = nonzero_indices[indices]
    x_coords = nonzero_indices[:, 0].cpu().numpy()
    y_coords = nonzero_indices[:, 1].cpu().numpy()
    z_coords = nonzero_indices[:, 2].cpu().numpy()

    # Поворот T на 90° вокруг X-оси: Z → Y, Y → -Z
    x_rotated = x_coords
    y_rotated = z_coords
    z_rotated = -y_coords

    # Дополнительный поворот на 180° вокруг X-оси: Y → -Y, Z → -Z
    x_final_rotated = -z_coords
    y_final_rotated = x_coords
    z_final_rotated = y_coords
 

    # Масштабирование T, чтобы соответствовать размеру backprojection
    # backprojection: X: 70 мм, Y: 35 мм, Z: 200 мм
    # T: X, Y, Z: 0..127 индексов, но после create_output_model масштаб меньше
    # Оценим реальный диапазон T после поворота
    x_range = x_final_rotated.max() - x_final_rotated.min()
    y_range = y_final_rotated.max() - y_final_rotated.min()
    z_range = z_final_rotated.max() - z_final_rotated.min()

    scale_x = 70.0 / x_range if x_range != 0 else 1.0
    scale_y = 35.0 / y_range if y_range != 0 else 1.0
    scale_z = 200.0 / z_range if z_range != 0 else 1.0

    x_scaled = x_final_rotated * scale_x
    y_scaled = y_final_rotated * scale_y
    z_scaled = z_final_rotated * scale_z

    # Центрируем T относительно backprojection
    x_center = (x_mm.max() + x_mm.min()) / 2
    y_center = (y_mm.max() + y_mm.min()) / 2
    z_center = (z_mm.max() + z_mm.min()) / 2

    x_final = x_scaled + x_center - (x_scaled.max() + x_scaled.min()) / 2
    y_final = y_scaled + y_center - (y_scaled.max() + y_scaled.min()) / 2
    z_final = z_scaled + z_center - (z_scaled.max() + z_scaled.min()) / 2
# Масштабирование T, чтобы соответствовать размеру backprojection
    x_range = x_final_rotated.max() - x_final_rotated.min()
    y_range = y_final_rotated.max() - y_final_rotated.min()
    z_range = z_final_rotated.max() - z_final_rotated.min()

    scale_x = 70.0 / x_range if x_range != 0 else 1.0
    scale_y = 35.0 / y_range if y_range != 0 else 1.0
    scale_z = 200.0 / z_range if z_range != 0 else 1.0

    x_scaled = x_final_rotated * scale_x
    y_scaled = y_final_rotated * scale_y
    z_scaled = z_final_rotated * scale_z

    # Уточнённое центрирование T относительно backprojection
    x_center = (x_mm.max() + x_mm.min()) / 2
    y_center = (y_mm.max() + y_mm.min()) / 2
    z_center = (z_mm.max() + z_mm.min()) / 2

    x_final = x_scaled + x_center - (x_scaled.max() + x_scaled.min()) / 2
    y_final = y_scaled + y_center - (y_scaled.max() + y_scaled.min()) / 2
    z_final = z_scaled + z_center - (z_scaled.max() + z_scaled.min()) / 2

    fig3.add_trace(go.Scatter3d(
        x=x_final, y=y_final, z=z_final,
        mode='markers',
        marker=dict(size=2, color='green', opacity=0.5),
        name='Original 3D Target (Rotated & Scaled)'
    ))

    fig3.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="Comparison: Reconstruction vs Original 3D Target"
    )
    fig3.show()

# # Трансформация
# images_transform = T.Compose([
#     T.ToTensor(),
#     T.Resize((128, 128)),
#     T.Lambda(lambda x: x * 5.0),
# ])

# data = GeneratedDataset(images_dir, labels_dir, info_dir, images_transform=images_transform)
# sample_index = 0
# sample = data[sample_index]

projection1 = sample[0][0].cuda()
projection2 = sample[0][4].cuda()
theta = sample[2]
phi = [90 + sample[3][0], 90 + sample[3][1]]
correct_3d_geometry = sample[1].cuda()

print(f"Using angles: theta={theta}, phi={phi}")

visualize_angiograph(projection1, projection2, theta, phi, correct_3d_geometry, volume_shape=[256, 256, 256])


## **Dataset**

In [ ]:
X_data = []
N = 20
for i in range(N):
    sample = data[i]
    projection1 = sample[0][0]  # Оставляем на CPU
    projection2 = sample[0][4]
    theta = sample[2]
    phi = [90 + sample[3][0], 90 + sample[3][1]]
    volume_shape = [128, 128, 128]
    vol1, v_source1, v_sensor1, sensor_points1, directions1, proj_binary1 = create_shadow_cone(projection1, theta[0], phi[0], volume_shape)
    vol2, v_source2, v_sensor2, sensor_points2, directions2, proj_binary2 = create_shadow_cone(projection2, theta[1], phi[1], volume_shape)
    backprojection = (vol1 > 0) & (vol2 > 0)
    backprojection = backprojection.float()
    non_zero = torch.where(backprojection > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0].cpu().numpy()
        y = non_zero[1].cpu().numpy()
        z = non_zero[2].cpu().numpy()
    else:
        x = non_zero[0].cpu().numpy()
        y = non_zero[1].cpu().numpy()
        z = non_zero[2].cpu().numpy()

    vol = torch.zeros((128, 128, 128))
    for xi, yi, zi in zip(x, y, z):
        if 0 <= xi < 128 and 0 <= yi < 128 and 0 <= zi < 128:
            vol[xi, yi, zi] = 1.0
    X_data.append(vol.unsqueeze(0))

X_data = torch.stack(X_data)

Y_data = []
for i in range(N):
    sample = data[i]
    correct_3d_geometry = sample[1].cpu()
    target_volume = create_output_model(correct_3d_geometry, volume_shape=[128, 128, 128])
    Y_data.append(target_volume.unsqueeze(0))

Y_data = torch.stack(Y_data)

In [ ]:
def visualize_voxel_tensor(volume_tensor, title="3D Voxel Volume", max_points=5000):
    """
    Показывает 3D volume из [128, 128, 128] или [1, 128, 128, 128]
    """
    # Убираем размерность канала, если есть
    if volume_tensor.dim() == 4:
        volume_tensor = volume_tensor.squeeze(0)
    
    # Получаем индексы всех ненулевых вокселей
    nonzero = (volume_tensor > 0).nonzero(as_tuple=False)

    if nonzero.size(0) == 0:
        print("Нет ненулевых точек для отображения.")
        return

    # Ограничим число точек
    if nonzero.size(0) > max_points:
        indices = torch.randperm(nonzero.size(0))[:max_points]
        nonzero = nonzero[indices]

    # Получим координаты
    x = nonzero[:, 0].cpu().numpy()
    y = nonzero[:, 1].cpu().numpy()
    z = nonzero[:, 2].cpu().numpy()

    # Построим 3D scatter plot
    fig = go.Figure(data=[
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(
                size=2,
                color='blue',
                opacity=0.8
            )
        )
    ])

    fig.update_layout(
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode='cube'
        ),
        title=title,
        width=800,
        height=800,
        template="plotly_white"
    )


    fig.show()


In [ ]:
visualize_voxel_tensor(X_data[5])

In [ ]:
import torch.nn as nn

class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = self.conv_block(1,32)
        self.enc2 = self.conv_block(32,64)
        self.enc3 = self.conv_block(64,128)
        
        self.dec3 = self.up_conv(128,64)
        self.dec2 = self.up_conv(64,32)
        self.dec1 = nn.Conv3d(32,1, kernel_size = 1)

        self.pool = nn.MaxPool3d(2)

    def conv_block(self, in_channels, out_channels):
            return nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm3d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm3d(out_channels),
                nn.ReLU(inplace=True),
            )
    def up_conv(self, in_channels, out_channels):
            return nn.Sequential(
                nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2),
                nn.ReLU(inplace=True),
                self.conv_block(out_channels, out_channels)
            )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Decoder
        d3 = self.dec3(e3)
        d2 = self.dec2(d3 + e2)
        d1 = self.dec1(d2 + e1)

        return torch.sigmoid(d1)


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

X_train, X_test, Y_train, Y_test = train_test_split(
    X_data, Y_data, test_size=0.2, random_state=42
)

class VoxelDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

train_dataset = VoxelDataset(X_train, Y_train)
test_dataset = VoxelDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet3D().to(device)
criterion = nn.BCELoss()  # или nn.L1Loss() — ты можешь сравнить позже
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
num_epochs = 10
accumulation_steps = 4  # Накапливаем градиенты через 4 батча

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()  # Очищаем градиенты в начале эпохи

    for i, (X_batch, Y_batch) in enumerate(train_loader):
        X_batch = X_batch.to(device, non_blocking=True)
        Y_batch = Y_batch.to(device, non_blocking=True)

        output = model(X_batch)
        loss = criterion(output, Y_batch)
        loss = loss / accumulation_steps  # Нормализуем лосс
        loss.backward()  # Накапливаем градиенты

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()  # Обновляем веса
            optimizer.zero_grad()  # Очищаем градиенты
            torch.cuda.empty_cache()  # Очищаем память

        total_loss += loss.item() * accumulation_steps

    print(f"Epoch {epoch+1}: Loss {total_loss:.4f}")

In [ ]:
from random import randint

i = randint(0, output.shape[0] - 1)

predicted = output[i].cpu().squeeze(0)  # [1, D, H, W] → [D, H, W]
ground_truth = Y_batch[i].cpu().squeeze(0)
input_data = X_batch[i].cpu().squeeze(0)

visualize_voxel_tensor(input_data, title="Backprojection (Input)")
visualize_voxel_tensor(predicted, title="Predicted Output (U-Net)")
visualize_voxel_tensor(ground_truth, title="Ground Truth (Target)")


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from scipy.ndimage import zoom
import torch.nn.functional as F

# Настройка для избежания фрагментации памяти
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Мониторинг памяти
def print_memory_usage():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")

# Очистка памяти перед началом
torch.cuda.empty_cache()
print_memory_usage()

class Backprojection:
    def __init__(self, sid, pixel_spacing, volume_size, volume_spacing, dso=None, img_dim=128):
        self.sid = sid
        self.pixel_spacing = pixel_spacing
        self.volume_size = volume_size
        self.volume_spacing = volume_spacing
        self.dso = dso if dso is not None else sid * 0.75
        self.img_dim = img_dim
        self.principal_point = np.array([img_dim/2, img_dim/2])
        self.distance_detector_to_iso = sid - self.dso
        self.coord_system_changer = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])

    def get_projection_matrix(self, theta, phi):
        theta = float(theta)
        phi = float(phi)
        theta_rad = np.radians(theta)
        phi_rad = np.radians(phi)

        Rotation_AP1 = self.coord_system_changer @ np.array([
            [np.cos(theta_rad), -np.sin(theta_rad), 0],
            [np.sin(theta_rad), np.cos(theta_rad), 0],
            [0, 0, 1]
        ]) @ np.linalg.inv(self.coord_system_changer)

        Rotation_AP2 = self.coord_system_changer @ np.array([
            [1, 0, 0],
            [0, np.cos(phi_rad), np.sin(phi_rad)],
            [0, -np.sin(phi_rad), np.cos(phi_rad)]
        ]) @ np.linalg.inv(self.coord_system_changer)

        Rotation_AP3 = np.eye(3)
        R_total = Rotation_AP1 @ Rotation_AP2 @ Rotation_AP3

        v_sensor = R_total @ np.array([0, 0, self.distance_detector_to_iso])
        v_source = -v_sensor / self.distance_detector_to_iso * self.dso

        z_axis = (v_sensor - v_source) / np.linalg.norm(v_sensor - v_source)
        x_axis = np.array([1, 0, 0])
        if np.abs(np.dot(z_axis, x_axis)) > 0.99:
            x_axis = np.array([0, 1, 0])
        x_axis = x_axis - np.dot(x_axis, z_axis) * z_axis
        x_axis = x_axis / np.linalg.norm(x_axis)
        y_axis = np.cross(z_axis, x_axis)
        y_axis = y_axis / np.linalg.norm(y_axis)

        R = np.vstack((x_axis, y_axis, z_axis)).T
        t = -v_source
        Rt = np.hstack((R, t.reshape(3, 1)))

        focal_length = self.sid / self.pixel_spacing
        K = np.array([
            [focal_length, 0, self.principal_point[0]],
            [0, focal_length, self.principal_point[1]],
            [0, 0, 1]
        ])
        P = K @ Rt
        return P, v_source, v_sensor, R

def create_shadow_cone(projection, theta, phi, volume_shape=(128, 128, 128), sid=1200.0, pixel_spacing=1.4, dso=750.0, img_dim=128):
    D_x, D_y, D_z = volume_shape
    projection = projection.clone().detach()
    threshold = 1
    # Бинарная маска для визуализации
    projection_binary = (projection > 1).float()

    # Создаём воксельное пространство, расширяем по X и Y
    vol = torch.zeros(volume_shape, dtype=torch.float32, device=projection.device)
    volume_spacing = 0.35
    x = torch.linspace(-100.0, 100.0, D_x, device=projection.device)  # Расширяем X
    y = torch.linspace(-100.0, 100.0, D_y, device=projection.device)  # Расширяем Y
    z = torch.linspace(-200.0, 200.0, D_z, device=projection.device)  # Z от -200 до 200 мм

    # Получаем геометрию камеры
    backprojector = Backprojection(sid, pixel_spacing, volume_shape, volume_spacing, dso, img_dim)
    P, v_source, v_sensor, R = backprojector.get_projection_matrix(theta, phi)

    # Преобразуем в тензоры
    v_source = torch.tensor(v_source, dtype=torch.float32, device=projection.device)
    v_sensor = torch.tensor(v_sensor, dtype=torch.float32, device=projection.device)
    R = torch.tensor(R, dtype=torch.float32, device=projection.device)

    # Определяем координаты пикселей проекции на сенсоре
    u_coords, v_coords = torch.where(projection > 1)
    intensities = projection[u_coords, v_coords]

    # Преобразуем u, v в 3D-координаты на сенсоре
    u_physical = (u_coords.float() - img_dim/2) * pixel_spacing
    v_physical = (v_coords.float() - img_dim/2) * pixel_spacing
    sensor_points = torch.stack([
        u_physical,
        v_physical,
        torch.zeros_like(u_physical)
    ], dim=1)

    # Поворачиваем точки сенсора в глобальную систему координат
    sensor_points = (R @ sensor_points.T).T + v_sensor

    # Для каждой точки на сенсоре строим луч к источнику
    directions = v_source - sensor_points
    directions = directions / torch.norm(directions, dim=1, keepdim=True)

    # Проходим по лучу через воксельное пространство
    t_max = 3000.0  # Увеличиваем расстояние
    t_steps = torch.linspace(0, t_max, 3000, device=projection.device)  # Больше шагов

    for i in range(len(sensor_points)):
        points_along_ray = sensor_points[i] + directions[i] * t_steps[:, None]
        
        # Преобразуем точки в воксельные индексы
        x_idx = ((points_along_ray[:, 0] - (-100.0)) / (200.0 / (D_x - 1))).long()
        y_idx = ((points_along_ray[:, 1] - (-100.0)) / (200.0 / (D_y - 1))).long()
        z_idx = ((points_along_ray[:, 2] - (-200.0)) / (400.0 / (D_z - 1))).long()

        # Проверяем, что индексы в пределах объёма
        valid = (x_idx >= 0) & (x_idx < D_x) & (y_idx >= 0) & (y_idx < D_y) & (z_idx >= 0) & (z_idx < D_z)

        x_idx, y_idx, z_idx = x_idx[valid], y_idx[valid], z_idx[valid]

        # Заполняем воксели
        vol[x_idx, y_idx, z_idx] = intensities[i]

    return vol, v_source, v_sensor, sensor_points, directions, projection_binary

from typing import List
def create_output_model(output_3d_model: torch.Tensor, volume_shape: List[int], target_physical_range: dict):
    with torch.no_grad():
        output_3d_model = output_3d_model.clone()
        D_x, D_y, D_z = volume_shape

        # Извлекаем физические диапазоны
        x_min, x_max = target_physical_range['x']
        y_min, y_max = target_physical_range['y']
        z_min, z_max = target_physical_range['z']
        x_range_physical = x_max - x_min
        y_range_physical = y_max - y_min
        z_range_physical = z_max - z_min

        # Центрируем координаты
        mean_coords = output_3d_model[..., :3].mean(dim=(0, 1))
        output_3d_model[..., :3] -= mean_coords

        # Поворот на 90° вокруг X: Z → Y, Y → -Z
        x_coords = output_3d_model[..., 0]
        y_coords = output_3d_model[..., 2]
        z_coords = -output_3d_model[..., 1]

        # Поворот на 180° вокруг X: Y → -Y, Z → -Z
        x_rotated = -z_coords
        y_rotated = x_coords
        z_rotated = y_coords

        # Смена осей X ↔ Z
        x_temp = x_rotated
        x_rotated = z_rotated
        z_rotated = x_temp

        # Замена x = -z
        x_final_rotated = -z_rotated
        y_final_rotated = y_rotated
        z_final_rotated = x_rotated

        # Масштабируем координаты
        x_range = x_final_rotated.max() - x_final_rotated.min()
        y_range = y_final_rotated.max() - y_final_rotated.min()
        z_range = z_final_rotated.max() - z_final_rotated.min()

        scale_x = x_range_physical / x_range if x_range != 0 else 1.0
        scale_y = y_range_physical / y_range if y_range != 0 else 1.0
        scale_z = z_range_physical / z_range if z_range != 0 else 1.0

        x_scaled = x_final_rotated * scale_x
        y_scaled = y_final_rotated * scale_y
        z_scaled = z_final_rotated * scale_z

        # Преобразуем в индексы вокселей
        x_indices = ((x_scaled - x_min) / (x_range_physical / (D_x - 1))).int().clip(0, D_x - 1)
        y_indices = ((y_scaled - y_min) / (y_range_physical / (D_y - 1))).int().clip(0, D_y - 1)
        z_indices = ((z_scaled - z_min) / (z_range_physical / (D_z - 1))).int().clip(0, D_z - 1)

        # Обновляем координаты
        output_3d_model[..., 0] = x_indices
        output_3d_model[..., 1] = y_indices
        output_3d_model[..., 2] = z_indices

        T = torch.zeros(volume_shape, dtype=torch.float32, device=output_3d_model.device)

        for group in range(output_3d_model.shape[0]):
            x = output_3d_model[group, :, 0].long()
            y = output_3d_model[group, :, 1].long()
            z = output_3d_model[group, :, 2].long()
            R = output_3d_model[group, :, 3] * 10
            for i in range(len(x)):
                radius = int((R[i] * 12).clamp(2, 15))
                x_grid, y_grid, z_grid = torch.meshgrid(
                    torch.arange(-radius, radius + 1, dtype=torch.long, device=output_3d_model.device),
                    torch.arange(-radius, radius + 1, dtype=torch.long, device=output_3d_model.device),
                    torch.arange(-radius, radius + 1, dtype=torch.long, device=output_3d_model.device)
                )
                mask = (x_grid ** 2 + y_grid ** 2 + z_grid ** 2 <= radius ** 2)
                x_shifted = (x[i] + x_grid[mask]).long()
                y_shifted = (y[i] + y_grid[mask]).long()
                z_shifted = (z[i] + z_grid[mask]).long()

                valid = (x_shifted >= 0) & (x_shifted < D_x) & (y_shifted >= 0) & (y_shifted < D_y) & (z_shifted >= 0) & (z_shifted < D_z)
                x_valid = x_shifted[valid]
                y_valid = y_shifted[valid]
                z_valid = z_shifted[valid]

                insertion_index = T[x_valid, y_valid, z_valid] == 0
                average_index = T[x_valid, y_valid, z_valid] != 0

                T[x_valid, y_valid, z_valid] += R[i] * insertion_index
                avg = (R[i] + T[x_valid, y_valid, z_valid]) / 2
                T[x_valid, y_valid, z_valid] += (avg - T[x_valid, y_valid, z_valid]) * average_index

        return T

def get_3d_tensors(projection1, projection2, correct_3d_geometry, tensor_output_shape=[128, 128, 128]):
    with torch.no_grad():
        D_x, D_y, D_z = tensor_output_shape
        projection1 = zoom(projection1.cpu(), (D_x / projection1.shape[0], D_y / projection1.shape[1]), order=1)
        projection2 = zoom(projection2.cpu(), (D_x / projection2.shape[0], D_y / projection2.shape[1]), order=1)

        projection1 = torch.tensor(projection1).to(projection1.device)
        projection2 = torch.tensor(projection2).to(projection2.device)

        # Обновляем физический диапазон под Backprojection
        physical_range = {
            'x': (-100.0, 100.0),
            'y': (-100.0, 100.0),
            'z': (-200.0, 200.0)
        }

        T = create_output_model(
            correct_3d_geometry.to(projection1.device),
            volume_shape=tensor_output_shape,
            target_physical_range=physical_range
        )
        return T

def visualize_comparison(X, Y, volume_shape=(128, 128, 128)):
    D_x, D_y, D_z = volume_shape
    fig = go.Figure()

    # Backprojection
    non_zero = torch.where(X[0, 0] > 0)
    max_points = 5000
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1][indices].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2][indices].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0
    else:
        x = non_zero[0].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0

    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=2, color='purple', opacity=0.5),
        name='Backprojection (Input)'
    ))

    # Ground Truth
    non_zero = torch.where(Y[0, 0] > 0)
    if len(non_zero[0]) > max_points:
        indices = torch.randperm(len(non_zero[0]))[:max_points]
        x = non_zero[0][indices].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1][indices].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2][indices].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0
    else:
        x = non_zero[0].cpu().numpy() * (200.0 / (D_x - 1)) - 100.0
        y = non_zero[1].cpu().numpy() * (200.0 / (D_y - 1)) - 100.0
        z = non_zero[2].cpu().numpy() * (400.0 / (D_z - 1)) - 200.0

    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(size=2, color='green', opacity=0.5),
        name='Ground Truth'
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title="Comparison: Backprojection vs Ground Truth"
    )
    fig.show()

# Подготовка данных
N = 20
X_data = []
Y_data = []
volume_shape = [256, 256, 256]  # Для Backprojection
final_shape = [128, 128, 128]   # Для U-Net

for i in range(N):
    print(f"Sample {i+1}/{N}")
    print_memory_usage()
    sample = data[i]
    projection1 = sample[0][0].cuda()
    projection2 = sample[0][4].cuda()
    theta = sample[2]
    phi = [90 + sample[3][0], 90 + sample[3][1]]
    correct_3d_geometry = sample[1].cuda()

    # Создаём Backprojection
    vol1, _, _, _, _, _ = create_shadow_cone(projection1, theta[0], phi[0], volume_shape)
    vol2, _, _, _, _, _ = create_shadow_cone(projection2, theta[1], phi[1], volume_shape)
    backprojection = (vol1 > 0) & (vol2 > 0)
    backprojection = backprojection.float()

    # Уменьшаем Backprojection до 128x128x128
    backprojection_resized = F.interpolate(
        backprojection.unsqueeze(0).unsqueeze(0),
        size=final_shape,
        mode='nearest'
    ).squeeze()
    X_data.append(backprojection_resized.unsqueeze(0))

    # Создаём Ground Truth
    target_volume = get_3d_tensors(projection1, projection2, correct_3d_geometry, tensor_output_shape=final_shape)
    Y_data.append(target_volume.unsqueeze(0))

    del vol1, vol2, backprojection, backprojection_resized, target_volume
    torch.cuda.empty_cache()

X_data = torch.stack(X_data)
Y_data = torch.stack(Y_data)
print_memory_usage()
torch.cuda.empty_cache()

# Визуализируем первый пример для проверки
visualize_comparison(X_data[0:1], Y_data[0:1], volume_shape=final_shape)

# Модель
class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self.conv_block(1, 16)
        self.enc2 = self.conv_block(16, 32)
        self.enc3 = self.conv_block(32, 64)
        
        self.dec3 = self.up_conv(64, 32)
        self.dec2 = self.up_conv(32, 16)
        self.dec1 = nn.Conv3d(16, 1, kernel_size=1)
        self.pool = nn.MaxPool3d(2)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def up_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            self.conv_block(out_channels, out_channels)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d3 = self.dec3(e3)
        d2 = self.dec2(d3 + e2)
        d1 = self.dec1(d2 + e1)
        return torch.sigmoid(d1)

# Dice Loss
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = pred.contiguous().view(-1)
        target = target.contiguous().view(-1)
        intersection = (pred * target).sum()
        return 1 - ((2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth))

# Датасет и обучение
X_train, X_test, Y_train, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)

class VoxelDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

train_dataset = VoxelDataset(X_train, Y_train)
test_dataset = VoxelDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet3D().to(device)
criterion = DiceLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
num_epochs = 50

scaler = GradScaler()
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_idx, (X_batch, Y_batch) in enumerate(train_loader):
        print(f"Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader)}")
        print_memory_usage()
        X_batch = X_batch.to(device, non_blocking=True)
        Y_batch = Y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast():
            output = model(X_batch)
            loss = criterion(output, Y_batch)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        torch.cuda.empty_cache()
        print_memory_usage()

    print(f"Epoch {epoch+1}: Loss {total_loss:.4f}")

# Визуализация предсказания
model.eval()
with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        X_batch = X_batch.to(device)
        output = model(X_batch)
        output = (output > 0.5).float()
        break

fig = go.Figure()
non_zero = torch.where(output[0, 0] > 0)
max_points = 5000
if len(non_zero[0]) > max_points:
    indices = torch.randperm(len(non_zero[0]))[:max_points]
    x = non_zero[0][indices].cpu().numpy() * (200.0 / (128 - 1)) - 100.0
    y = non_zero[1][indices].cpu().numpy() * (200.0 / (128 - 1)) - 100.0
    z = non_zero[2][indices].cpu().numpy() * (400.0 / (128 - 1)) - 200.0
else:
    x = non_zero[0].cpu().numpy() * (200.0 / (128 - 1)) - 100.0
    y = non_zero[1].cpu().numpy() * (200.0 / (128 - 1)) - 100.0
    z = non_zero[2].cpu().numpy() * (400.0 / (128 - 1)) - 200.0

fig.add_trace(go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(size=2, color='purple', opacity=0.5),
    name='Predicted Output (U-Net)'
))

fig.update_layout(
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=1)
    ),
    title="Predicted Output (U-Net)"
)
fig.show()